# LeapQuant 教程 (3/5): LeapQuant 量化器 —— 基、双尺度 int8、补偿 (论文 eq.5–9)

前两本: 状态 = 账本; 量化 = 刻度尺 + 舍入。这本把两者接起来: **在窗口边界给账本拍一张
int8 快照**时, 怎么让这一下尽量不失真。

对应源码 `vllm_ascend/quantization/methods/mamba_state/leapquant/state.py` (PR1 #21120)。

核心观察: 账本虽然是满秩矩阵, 但**能量集中在少数方向上** (低秩倾向)。

| 步骤 | 函数 | 论文 | 干什么 |
|---|---|---|---|
| ① 起始基 | `start_basis` | — | 固定种子正交基 $q_0$ (跨设备逐位复现) |
| ② 补偿对 | `fit_compensators` | eq.5-6 | 一轮子空间迭代 + MGS → top-$r$ 方向对 $(q, u)$, **fp16 保存** |
| ③ 残差编码 | `quantize_residual` | eq.7 | 残差 $E = S - q^{\top}u$ 双尺度 int8: 列 RMS $s_k$ × 行 absmax $s_v$ |
| ④ 重建 | `reconstruct_state` | eq.8-9 | $\hat{S} = q^{\top}u + \mathrm{codes}\cdot(s_k/127)\cdot s_v$ |

直觉: **主方向用少量 fp16 精确记** (top-$r$ 外积), **剩下的小残差才走 int8** —— 而残差的
动态范围小, 刻度尺细, 舍入误差小。

In [1]:
import sys, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发树 (分支 leapquant-triton-m3)

try:
    import torch_npu  # noqa: F401 — 注册 npu 后端
    DEV = torch.device("npu") if torch.npu.is_available() else torch.device("cpu")
except Exception:
    DEV = torch.device("cpu")

import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("vllm").setLevel(logging.ERROR)

torch.manual_seed(20261110)
print(f"torch = {torch.__version__}   device = {DEV}")

torch = 2.10.0+cpu   device = npu


/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


In [2]:
H, K, V = 8, 128, 128        # Qwen3.5-0.8B 单卡 GDN 头几何 (128×128)
B, WINDOW, R = 4, 16, 4      # 批 / 窗口 / 补偿秩 (论文 8-bit 最优)
SCALE = K ** -0.5

def gdn_inputs(B, H, T, K=K, V=V, dev=DEV):
    q = torch.nn.functional.normalize(torch.randn(B, H, T, K, device=dev), dim=-1)
    k = torch.nn.functional.normalize(torch.randn(B, H, T, K, device=dev), dim=-1)
    v = torch.randn(B, H, T, V, device=dev)
    g = -torch.rand(B, H, T, device=dev) * 0.1       # log 域衰减率 ∈ [-0.1, 0)
    beta = torch.rand(B, H, T, device=dev).sigmoid() # delta 门控
    return q, k, v, g, beta

def gdn_step(h, q, k, v, g, beta):
    """一步递推 (逐项拆解见教程 1): 衰减 → delta 修正 → 外积更新。"""
    h = h * g.exp()[..., None, None]
    v_adj = (v - torch.einsum("bhvk,bhk->bhv", h, k)) * beta[..., None]
    return h + torch.einsum("bhv,bhk->bhvk", v_adj, k)

def gdn_out(h, q):
    return torch.einsum("bhvk,bhk->bhv", h, q) * SCALE

def trajectory(inputs, window=None, boundary_fn=None, every_step_fn=None):
    """跑 T 步; 可选在窗口边界 / 每步对状态做一次变换 (量化实验的注入点)。"""
    q, k, v, g, beta = inputs
    B, H, T = q.shape[0], q.shape[1], q.shape[2]
    h = torch.zeros(B, H, V, K, device=q.device)
    outs = []
    for t in range(T):
        h = gdn_step(h, q[:, :, t], k[:, :, t], v[:, :, t], g[:, :, t], beta[:, :, t])
        if every_step_fn is not None:
            h = every_step_fn(h)
        elif boundary_fn is not None and (t + 1) % window == 0:
            h = boundary_fn(h)
        outs.append(gdn_out(h, q[:, :, t]))
    return h, torch.stack(outs, dim=2)

print("GDN 模拟器就绪 (完整拆解见教程 1)")

GDN 模拟器就绪 (完整拆解见教程 1)


## 1. 能量真的集中吗? —— 看一眼状态的奇异值谱

先滚一个真实窗口 (16 token) 的边界状态, 对每个 `[V,K]` 矩阵做 SVD, 看前 $r$ 个奇异值
占多少能量。SVD 的 top-$r$ 是**理论最优**的低秩捕获上界。

In [3]:
from vllm_ascend.quantization.methods.mamba_state.leapquant.state import (
    QMAX, start_basis, fit_compensators, quantize_residual,
    reconstruct_state, leap_window_flush,
)

inp1 = gdn_inputs(B, H, WINDOW)           # 一个窗口的输入
S, _ = trajectory(inp1)                   # fp32 递推 → 边界状态 [B,H,V,K]
blocks = S.reshape(-1, V, K)              # [N=B*H, V, K]

sv = torch.linalg.svdvals(blocks.double())
e_top = ((sv[:, :R] ** 2).sum(-1) / (sv ** 2).sum(-1))
print(f"top-{R} 奇异值能量占比:  均值 {e_top.mean():.4f}   最小 {e_top.min():.4f}   最大 {e_top.max():.4f}")
print(f"→ 平均 {e_top.mean():.0%} 的能量压在 {R}/{V} 个方向上 —— 低秩压缩有物理基础 (SVD 上界)")

INFO 11-10 21:29:39 [__init__.py:52] Available plugins for group vllm.platform_plugins:
INFO 11-10 21:29:39 [__init__.py:54] - ascend -> vllm_ascend:register
INFO 11-10 21:29:39 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 11-10 21:29:39 [__init__.py:272] Platform plugin ascend is activated
INFO 11-10 21:29:39 [platform.py:85] Breakable CUDAGraph on Ascend is opt-in; using VLLM_USE_BREAKABLE_CUDAGRAPH=0.


[W1110 21:29:46.074236309 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


top-4 奇异值能量占比:  均值 0.4767   最小 0.4256   最大 0.5480
→ 平均 48% 的能量压在 4/128 个方向上 —— 低秩压缩有物理基础 (SVD 上界)


[W1110 21:29:46.452165388 ToKernelNpu.cpp:40] Warning: Device do not support double dtype now, dtype cast replace with float. (function operator())


SVD 又贵又不可复现 (设备/库相关), 不能用在在线路径上。LeapQuant 用**一轮子空间迭代**
去追这个 top-$r$ 子空间: 拿固定起始基 $q_0$ 对状态做一次幂迭代, 再正交化。

## 2. 起始基 `start_basis`: 固定种子, 逐位复现

起始基必须与执行历史无关 (同一模型任何时刻做 flush, 基都一样), 否则相邻两次 flush 的
基会"漂移", 误差不可控。验证: 调两次, 结果逐位相等, 且列列正交。

In [4]:
q0  = start_basis(V, R, device=DEV, dtype=torch.float32)
q0b = start_basis(V, R, device=DEV, dtype=torch.float32)
orth = (q0.T @ q0 - torch.eye(R, device=DEV)).abs().max()
print(f"q0 {tuple(q0.shape)}  正交性 max|q0ᵀq0−I| = {orth:.1e}  固定种子逐位复现: {torch.equal(q0, q0b)}")

q0 (128, 4)  正交性 max|q0ᵀq0−I| = 6.6e-07  固定种子逐位复现: True


## 3. 补偿对 `fit_compensators` (eq.5-6): 一轮子空间迭代追 top-r

$u, q$: $r$ 对向量, $q^{\top}u \approx S$ 的 top-$r$ 部分, **fp16 存储** (不走 int8)。
看看一轮幂迭代能追到 SVD 上界的几成。

In [5]:
u, q = fit_compensators(blocks, q0, R)
lowrank = torch.einsum("nrv,nrk->nvk", q.float(), u.float())
e_lr = ((lowrank ** 2).flatten(1).sum(-1) / (blocks ** 2).flatten(1).sum(-1)).mean().item()
print(f"top-{R} 子空间能量占比:  一轮子空间迭代捕获 {e_lr:.4f}   SVD top-{R} 上界 {e_top.mean():.4f}")
print(f"→ 一轮迭代拿到 SVD 上界的 {e_lr/e_top.mean():.0%}, 而代价只有一次小矩阵乘 + MGS")

top-4 子空间能量占比:  一轮子空间迭代捕获 0.3690   SVD top-4 上界 0.4767
→ 一轮迭代拿到 SVD 上界的 77%, 而代价只有一次小矩阵乘 + MGS


## 4. 残差双尺度 int8 `quantize_residual` (eq.7)

把 top-$r$ 部分"记账"后, 残差 $E = S - q^{\top}u$ 才走 int8。关键技巧是**两把刻度尺**:

- $s_k$ (每列, RMS): 残差的列能量差异大, 列各有各的尺;
- $s_v$ (每行, absmax): 行内动态范围再兜一层底。

codes 只在"基坐标系 + 双尺度归一"后的窄动态范围里取整 —— 所以同一套 int8 格点,
用得更省。

In [6]:
codes, s_k, s_v = quantize_residual(blocks, u, q)
print(f"codes ∈ [{codes.min()}, {codes.max()}]   mean|codes| = {codes.float().abs().mean():.1f}")
print(f"±127 钳制占比 {(codes.float().abs() >= QMAX).float().mean():.2%}   (理想情况 ≈ 0: 格点够用)")
print(f"s_k ∈ [{s_k.min():.3e}, {s_k.max():.3e}]   s_v ∈ [{s_v.min():.3e}, {s_v.max():.3e}]")

codes ∈ [-127, 127]   mean|codes| = 40.1
±127 钳制占比 0.81%   (理想情况 ≈ 0: 格点够用)
s_k ∈ [1.915e-01, 4.408e-01]   s_v ∈ [3.632e-01, 2.305e+00]


## 5. 重建 + 单次误差对比: 2.4× 从哪来 (eq.8-9)

重建 $\hat{S} = q^{\top}u + \mathrm{codes}\cdot\frac{s_k}{127}\cdot s_v$, 和教程 2 的
per-tensor int8 在**同一个状态**上比一次 round-trip。

In [7]:
S_hat = reconstruct_state(codes, s_k, s_v, u, q)
err_leap = ((S_hat - blocks).norm() / blocks.norm()).item()
s_pt = blocks.abs().amax(dim=(-2, -1), keepdim=True) / QMAX
err_pt = ((torch.round(blocks / s_pt) * s_pt - blocks).norm() / blocks.norm()).item()
print(f"单次 round-trip 相对误差 (同一状态):")
print(f"  LeapQuant (基+双尺度+补偿): {err_leap:.3%}")
print(f"  per-tensor int8 (教程 2):   {err_pt:.3%}   → LeapQuant 好 {err_pt/err_leap:.1f}×")

单次 round-trip 相对误差 (同一状态):
  LeapQuant (基+双尺度+补偿): 0.458%
  per-tensor int8 (教程 2):   1.098%   → LeapQuant 好 2.4×


## 6. 打包: `leap_window_flush` = 以上四步流水线

`leap_window_flush(S)` 把 ①–④ 打包成"给状态拍一张 int8 快照再还原"的一次变换 ——
这就是窗口边界处发生的一切 (教程 4 讲为什么在边界做)。顺手验证一个好消息:
**连续 flush 两次, 误差不翻倍** (量化误差对自身是收敛的, 不是自由累积的)。

In [8]:
S1 = leap_window_flush(blocks)                       # 第 1 次
S2 = leap_window_flush(S1)                           # 第 2 次 (对已量化状态再量化)
e1 = ((S1 - blocks).norm() / blocks.norm()).item()
e2 = ((S2 - blocks).norm() / blocks.norm()).item()
print(f"flush 1 次: {e1:.3%}    flush 2 次: {e2:.3%}    (误差有界, 不随 flush 次数线性增长)")

flush 1 次: 0.458%    flush 2 次: 0.521%    (误差有界, 不随 flush 次数线性增长)


## 小结 (教程 3)

- 状态能量集中 (top-4 平均 ~37%, SVD 上界 ~48%) → 低秩结构真实存在;
- `start_basis` 固定种子逐位复现; 一轮子空间迭代追到 SVD 上界的 ~8 成;
- **fp16 补偿 top-r + 双尺度 int8 残差** → 单次 round-trip 0.458%, 比朴素 int8 (1.098%) 好 2.4×;
- 连续 flush 误差有界。

→ **教程 4**: 把这个量化器放到**窗口边界**节奏上, 看 256 步的误差曲线, 并扫描 window 与 r。